In [ ]:
import arxiv
from langgraph.graph import StateGraph , START , END
from typing import TypedDict


urls = []

client = arxiv.Client()
search = arxiv.Search(
    query="Attention",
    max_results=5,
    sort_by=arxiv.SortCriterion.SubmittedDate
)

for result in client.results(search):
    # Build the PDF URL manually
    pdf_url = result.entry_id.replace("abs", "pdf") + ".pdf"
    urls.append(pdf_url)

print(urls)



In [ ]:
class ResearchState(TypedDict):
  query : str
  paper_info : str
  summary : str
  weak_evidence : str
  logical_inconsistency : str
  missing_perspective : str
  report : str

In [ ]:
from langchain_community.document_loaders import PyPDFLoader

paper_info = []
for url in urls:
  loader = PyPDFLoader(url)
  docs = loader.load()
  paper_info.extend(docs)


In [ ]:
paper_info[0].page_content

full_text = " ".join(doc.page_content for doc in paper_info)

In [ ]:
full_text

In [ ]:
from langchain_groq import ChatGroq
from dotenv import load_dotenv
import os

load_dotenv()

llm = ChatGroq(
    model="meta-llama/llama-4-maverick-17b-128e-instruct",
    api_key=os.getenv("GROQ_API_KEY"),
)


In [ ]:
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
prompt = PromptTemplate(

    template= """ You are an expert academic researcher. I will provide you with text excerpts from multiple research papers on a specific topic.
Your task is to:

Summarize the papers concisely.

Synthesize the given text into a coherent overall summary that captures the state of knowledge in this research area.

Present the final output with the following structure:

Topic Overview: (2–3 lines explaining the main focus area)

Paper-wise Summary: (brief, clear summaries for each paper)

{full_text}
""",
input_variables=['full_text'] 
)
 
parser = StrOutputParser()
chain = prompt | llm | parser

In [ ]:
summary = chain.invoke({full_text[:6000]})
print(summary)

In [ ]:
import tempfile
import requests
def paper_info(state : ResearchState) -> ResearchState:
    urls = []

    client = arxiv.Client()

    search = arxiv.Search(
        query =  state['query'],
        max_results=5,
        sort_by=arxiv.SortCriterion.SubmittedDate
    )

    for result in client.results(search):
        if result.pdf_url and result.pdf_url.startswith("http"):
            urls.append(result.pdf_url)
    
    if not urls:
        return {"paper_info": "No valid PDFs found for this query."}

    paper_info = []

    for url in urls:
        response = requests.get(url)
        tmp = tempfile.NamedTemporaryFile(delete=False, suffix=".pdf")
        tmp.write(response.content)
        tmp.flush()
        loader = PyPDFLoader(tmp.name)
        docs = loader.load()
        paper_info.extend(docs)

    full_text = " ".join(doc.page_content for doc in paper_info)

    return {'paper_info' : full_text}


   

In [ ]:
def summary(state : ResearchState) -> ResearchState:

    paper_info = state['paper_info']

    llm = ChatGroq(
    model="meta-llama/llama-4-maverick-17b-128e-instruct",
    api_key= os.getenv("GROQ_API_KEY")
    )

    prompt = PromptTemplate(
    template= """ You are an expert academic researcher. I will provide you with text excerpts from multiple research papers on a specific topic.
        
    Your task is to:
    
    Summarize the papers concisely.
    
    Synthesize the given text into a coherent overall summary that captures the state of knowledge in this research area.
    
    Present the final output with the following structure:
    
    Topic Overview: (2–3 lines explaining the main focus area)
    
    Paper-wise Summary: (brief, clear summaries for each paper)
    
    {full_text}
    """,
    input_variables=['full_text']
    )

    parser = StrOutputParser()
    chain = prompt | llm | parser

    result = chain.invoke({"full_text" : full_text[:6000]})

    return {'summary' : result}



In [ ]:

from langchain_classic.output_parsers import StructuredOutputParser , ResponseSchema
from langchain_core.prompts import PromptTemplate

In [ ]:
def evaluate(state : ResearchState) -> ResearchState:
    response_schemas = [
    ResponseSchema(name="weak_evidence", description="Describe any points in the summary where evidence or data support is weak or insufficient."),
    ResponseSchema(name="logical_inconsistency", description="Highlight any contradictions, unclear reasoning, or illogical conclusions present."),
    ResponseSchema(name="missing_perspective" , description="Mention any important perspectives, populations, or methodologies that are missing or overlooked.")
    ]
    output_parser = StructuredOutputParser.from_response_schemas(response_schemas)
    format_instructions = output_parser.get_format_instructions()

    prompt = PromptTemplate(
    template=""""
    You are an academic research critic.
    Given the following research summary, identify and extract key weaknesses.
    
    Give weak_evidence , logical_inconsistency and missing_perspective.
    {format_instructions}
    Research Summary:
    {summary}
    """,
    input_variables=['summary'],
    partial_variables = {"format_instructions" : format_instructions}
    )
    
    chain = prompt | llm | output_parser

    result = chain.invoke({"summary" : state['summary']})

    return {'weak_evidence' : result['weak_evidence'] , 'logical_inconsistency' : result['logical_inconsistency'] , 'missing_perspective' : result['missing_perspective']}

    





In [ ]:
def get_report(state : ResearchState) -> ResearchState:

    summary = state['summary']
    weak_evidence = state['weak_evidence']
    logical_inconsistency = state['logical_inconsistency']
    missing_perspective = state['missing_perspective']
    query = state['query']

    prompt = PromptTemplate(
        template= """
generate a report based on the following summary : {summary}
and the topic of this summary is {query}

Here are the provided critics for the above summary:
weak_evidence - {weak_evidence}
logical_inconsistency - {logical_inconsistency}
missing_perspective - {missing_perspective} 
""",
input_variables=['summary' , 'query' , 'weak_evidence' , 'logical_inconsistency','missing_perspective' ]
    )

    chain = prompt | llm | parser

    report = chain.invoke({"summary" : state['summary'] ,  'query' : state['query'] , 'weak_evidence' : state['weak_evidence'] , 'logical_inconsistency' : state['logical_inconsistency'] ,'missing_perspective' : state['missing_perspective']})

    
    return {'report' : report }


In [ ]:
# response_schemas = [
#     ResponseSchema(name="weak_evidence", description="Describe any points in the summary where evidence or data support is weak or insufficient."),
#     ResponseSchema(name="logical_inconsistency", description="Highlight any contradictions, unclear reasoning, or illogical conclusions present."),
#     ResponseSchema(name="missing_perspective" , description="Mention any important perspectives, populations, or methodologies that are missing or overlooked.")
#     ]
# output_parser = StructuredOutputParser.from_response_schemas(response_schemas)
# format_instructions = output_parser.get_format_instructions()

# prompt = PromptTemplate(
# template=""""
#     You are an academic research critic.
#     Given the following research summary, identify and extract key weaknesses.
    
#     Give weak_evidence , logical_inconsistency and missing_perspective.
#     {format_instructions}
#     Research Summary:
#     {summary}
#     """,
#     input_variables=['summary'],
#     partial_variables={"format_instructions" : format_instructions}
#     )
    
# chain = prompt | llm | output_parser

# result = chain.invoke({summary})

# print(result)


In [ ]:
# result['weak_evidence']

In [ ]:
graph = StateGraph(ResearchState)

graph.add_node('paper_info' , paper_info)
graph.add_node('summary' , summary)
graph.add_node('evaluate' , evaluate)
graph.add_node('report' , get_report)

graph.add_edge(START , 'paper_info')
graph.add_edge('paper_info' , 'summary')
graph.add_edge('summary' , 'evaluate')
graph.add_edge('evaluate' , 'report')
graph.add_edge('report' , END)



In [ ]:
workflow = graph.compile()

In [ ]:
workflow

In [ ]:
initial_state = {
    'query' : "papers about attention mechanism"
}

In [ ]:
workflow.invoke(initial_state)